# 01 · Load the code excerpts into `bluerridge_agents.code_excerpts`

Reads the 52 markdown clause summaries in `code_excerpts/` (YAML front-matter plus a short plain-language body) and upserts them into a Unity Catalog Delta table. That table is the source for the Vector Search index built in `02_create_endpoint_and_index.ipynb`.

> **Disclaimer.** The five project citation labels are fixed, illustrative labels and clause numbers are not authoritative (in the real ASME B16.5, pressure-temperature ratings are in section 2, not 4.2). ASME and IBC entries are plain-language paraphrases, never quotes of the copyrighted codes. See `code_excerpts/README.md`.

**How the notebook reads the files.** It reads `code_excerpts/` *relative to its own folder*. A `databricks bundle deploy` syncs that folder to the bundle's workspace files next to this notebook, and a notebook's working directory is its own folder, so the load is reproducible with no manual upload. (For the committed evidence run, the same folder was imported to a scratch workspace folder beside a copy of the notebook.)

**Table design.** `citation_id` is the primary key. `text` is `title + ". " + body`, so the embedding sees the clause heading as well as the body. Change Data Feed is enabled because a Delta Sync Vector Search index needs it. The load is an idempotent MERGE (insert, update, and delete rows whose file is gone), so re-running keeps the table equal to the folder.

In [1]:
import re
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import Row

dbutils.widgets.text("catalog", "mfg_mc_se_sa")
dbutils.widgets.text("schema", "bluerridge_agents")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE = f"{CATALOG}.{SCHEMA}.code_excerpts"
EXCERPT_DIR = Path.cwd() / "code_excerpts"  # a notebook's working directory is its own folder

FIELDS = ["citation_id", "citation_label", "title", "source", "is_paraphrase", "related_flaw_type"]


def parse_excerpt(path):
    """Split `---` front-matter (flat `key: value` lines) from the body. No YAML library needed."""
    raw = path.read_text(encoding="utf-8")
    assert raw.startswith("---\n"), f"{path.name}: no front-matter"
    _, fm, body = raw.split("---\n", 2)
    meta = {}
    for line in fm.strip().splitlines():
        key, _, val = line.partition(":")
        val = val.strip()
        if len(val) >= 2 and val[0] == val[-1] == '"':
            val = val[1:-1]
        meta[key.strip()] = val
    meta["is_paraphrase"] = meta["is_paraphrase"].lower() == "true"
    missing = [f for f in FIELDS if meta.get(f) in (None, "")]
    assert not missing, f"{path.name}: missing {missing}"
    body = re.sub(r"\s+", " ", body).strip()
    assert body, f"{path.name}: empty body"
    meta["text"] = f"{meta['title']}. {body}"
    return meta


files = sorted(p for p in EXCERPT_DIR.glob("*.md") if p.name.lower() != "readme.md")
rows = [parse_excerpt(p) for p in files]
ids = [r["citation_id"] for r in rows]
assert len(ids) == len(set(ids)), "citation_id must be unique"
print(f"{len(files)} excerpt files parsed from {EXCERPT_DIR.name}/")
print("disclaimer: citation labels are illustrative project labels, clause numbers are not authoritative; ASME/IBC entries are paraphrases")

52 excerpt files parsed from code_excerpts/
disclaimer: citation labels are illustrative project labels, clause numbers are not authoritative; ASME/IBC entries are paraphrases


In [2]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
  citation_id        STRING    NOT NULL COMMENT 'Stable slug; primary key and the Vector Search primary key',
  citation_label     STRING    COMMENT 'Citation string as written in compliance_findings.citation (exact-match lookup key for the five primary entries)',
  title              STRING,
  source             STRING    COMMENT 'ASME B16.5 | ASME B31.3 | OSHA 29 CFR 1910 | IBC',
  is_paraphrase      BOOLEAN   COMMENT 'true = plain-language paraphrase, not quoted code text',
  related_flaw_type  STRING    COMMENT 'Planted flaw type this clause explains, or none',
  text               STRING    COMMENT 'title + body; the column that is embedded',
  loaded_at          TIMESTAMP,
  CONSTRAINT code_excerpts_pk PRIMARY KEY (citation_id)
)
COMMENT 'Plain-language code-clause summaries (paraphrased; illustrative labels) that explain compliance findings. Source of the code_excerpts_vs Vector Search index.'
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

loaded_at = datetime.now(timezone.utc)
df = spark.createDataFrame(
    [Row(**{k: r[k] for k in FIELDS + ["text"]}, loaded_at=loaded_at) for r in rows]
).select("citation_id", "citation_label", "title", "source", "is_paraphrase", "related_flaw_type", "text", "loaded_at")
df.createOrReplaceTempView("code_excerpts_stage")

spark.sql(f"""
MERGE INTO {TABLE} t
USING code_excerpts_stage s
ON t.citation_id = s.citation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
spark.sql(f"ALTER TABLE {TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")  # also covers a table made earlier without it
print(f"merged {len(rows)} rows into {TABLE}")

merged 52 rows into mfg_mc_se_sa.bluerridge_agents.code_excerpts


## Checks: row count equals file count, change data feed is on, the five primary labels are present exactly

In [3]:
n = spark.table(TABLE).count()
print(f"table rows = {n}, excerpt files = {len(files)}")
assert n == len(files), "table row count must equal the number of excerpt files"

props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {TABLE}").collect()}
print("delta.enableChangeDataFeed =", props.get("delta.enableChangeDataFeed"))
assert props.get("delta.enableChangeDataFeed") == "true"

PRIMARY = ["ASME B16.5 §4.2", "OSHA 1910.147", "IBC 1005.1", "ASME B31.3 §341", "ASME B31.3 §322"]
have = {r["citation_label"] for r in spark.table(TABLE).collect()}
missing = [p for p in PRIMARY if p not in have]
print("primary labels missing:", missing)
assert not missing
bad = spark.sql(f"SELECT citation_id FROM {TABLE} WHERE (source LIKE 'ASME%' OR source LIKE 'IBC%') AND NOT is_paraphrase").count()
print("ASME/IBC rows not marked paraphrase:", bad)
assert bad == 0

display(spark.sql(f"""
SELECT source, is_paraphrase, COUNT(*) AS excerpts
FROM {TABLE} GROUP BY source, is_paraphrase ORDER BY source"""))

table rows = 52, excerpt files = 52
delta.enableChangeDataFeed = true
primary labels missing: []
ASME/IBC rows not marked paraphrase: 0


source,is_paraphrase,excerpts
ASME B16.5,True,10
ASME B31.3,True,19
IBC,True,12
OSHA 29 CFR 1910,True,11


In [4]:
display(spark.sql(f"""
SELECT citation_id, citation_label, source, is_paraphrase, related_flaw_type, LEFT(text, 110) AS text_start
FROM {TABLE} WHERE citation_label IN ('ASME B16.5 §4.2','OSHA 1910.147','IBC 1005.1','ASME B31.3 §341','ASME B31.3 §322')
ORDER BY citation_label"""))

citation_id,citation_label,source,is_paraphrase,related_flaw_type,text_start
asme-b16-5-4-2,ASME B16.5 §4.2,ASME B16.5,True,flange_rating_mismatch,Flange pressure class must suit design pressure and temperature. Every flange on a line must carry a pressure
asme-b31-3-322,ASME B31.3 §322,ASME B31.3,True,missing_pressure_relief_notation,Pressure-relief requirements. Piping and equipment that could see pressure above its design limit must be prot
asme-b31-3-341,ASME B31.3 §341,ASME B31.3,True,weld_joint_spec_mismatch,"Weld examination and joint requirements. Welds in process piping must be examined to a defined extent, and the"
ibc-1005-1,IBC 1005.1,IBC,True,egress_width_under_code,"Minimum width of the means of egress. The width of each egress component, such as a door, corridor, stair or e"
osha-1910-147,OSHA 1910.147,OSHA 29 CFR 1910,True,missing_loto_tag,Control of hazardous energy: lockout and tagout. Before anyone services or maintains a machine or piece of equ
